# Tutorial 3: GPU Kernel Memory Topology & 8-GPU Distributed Scaling (16,384 Envs)

Analyze how **Contact-Centric Link-Bucketed Scatter**, **Continuous Frame Arenas**, and **ZeRO-2 Sharding** reduce per-GPU VRAM and NVLink communication overhead across 8 GPUs.

In [ ]:
import matplotlib.pyplot as plt
from bimodal_tactile.distributed import DistributedMemoryBudget

budget = DistributedMemoryBudget(num_gpus=8, envs_per_gpu=2048, n_taxels=1140)
stats = budget.compute_breakdown_gib()
for k, v in stats.items():
    print(f'{k:<28}: {v:8.2f}')

labels = ['Hall Rollout Buffer', 'VBT Image Buffer', '1.5B Model + Adam']
naive_vals = [stats['naive_hall_rollout_gib'], stats['naive_vbt_fp32_gib'], stats['naive_model_opt_gib']]
opt_vals = [stats['arena_hall_rollout_gib'], stats['compact_vbt_int8_gib'], stats['zero2_model_opt_gib']]

x = range(len(labels))
plt.figure(figsize=(8, 4.5))
plt.bar([i - 0.18 for i in x], naive_vals, width=0.35, label='Naive Implementation (GiB)', color='#d95f02')
plt.bar([i + 0.18 for i in x], opt_vals, width=0.35, label='Optimized Arena + Int8 + ZeRO-2 (GiB)', color='#1b9e77')
plt.xticks(list(x), labels)
plt.ylabel('Per-GPU VRAM (GiB)')
plt.title('Per-GPU VRAM Footprint at 16,384 Total Parallel Envs (8x GPU)')
plt.legend(); plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.show()